# Airflow

A comprehensive guide to Airflow for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Apache Airflow is a **workflow orchestration platform** for authoring, scheduling, and monitoring data and ML pipelines as code.

### What is it?

- A **Python-based orchestrator** where you define workflows as DAGs (Directed Acyclic Graphs) of tasks.  
- A **scheduler + web UI + workers** that execute those tasks on a regular cadence or on demand.  
- A rich ecosystem of **operators** and **hooks** for databases, cloud services, warehouses, and ML platforms.

### Why use it?

Key benefits of using Airflow:

- **Pipelines as code**: Workflows are Python files checked into version control.  
- **Rich ecosystem**: Hundreds of operators for ETL, ML, and cloud services.  
- **Mature scheduling & backfill**: Robust cron-like scheduling, retries, and catchup.  
- **Observability**: Web UI, logs per task, Gantt charts, and alerting.

### When to use it?

Airflow is particularly useful when:

- You have **recurring batch workflows** (ETL, feature generation, model retraining).  
- You want **fine-grained control** over task dependencies and retry policies.  
- You need **centralized orchestration** for many teams’ data/ML jobs in one place.

## Key Features

### Core Capabilities of Airflow

| Feature | Description | Benefit |
|--------|-------------|---------|
| **DAGs as code** | Define workflows as Python DAGs with explicit dependencies. | Versionable, testable, reviewable workflows. |
| **Operators & Hooks** | Prebuilt tasks for common systems (SQL, S3, GCS, EMR, Databricks, Kubernetes, etc.). | Faster pipeline development with less boilerplate. |
| **Scheduler & Executor models** | Pluggable executors (Local, Celery, Kubernetes, etc.). | Scale from a single node to large clusters. |
| **Web UI** | Visualize DAGs, see logs, trigger runs, clear tasks. | Strong observability and manual control. |
| **Backfill & catchup** | Re-run historical intervals easily. | Fix data gaps and recompute features or models. |
| **Extensions ecosystem** | Providers packages for AWS, GCP, Azure, Databricks, etc. | Tight integration with cloud platforms and ML stacks. |

## Architecture Overview

At a high level, Airflow consists of a few core components:

```text
          +---------------------------+
          |       Web Server         |
          |   (Flask, RBAC, UI)      |
          +------------+--------------+
                       |
                       v
+-----------+   +------+-------+   +-------------------+
|  Users    |   |  Scheduler  |   |  Metadata DB       |
| (CLI/UI)  |   |  (DAG runs) |   |  (Postgres/MySQL)  |
+-----------+   +------+-------+   +---------+---------+
                       |                     ^
                       v                     |
                 +-----+---------------------+
                 |      Executors/Workers    |
                 |  (Local, Celery, K8s)     |
                 +---------------------------+
```

### Components

1. **Metadata Database**  
   Stores DAG definitions, task states, run history, and configuration.

2. **Scheduler**  
   Evaluates DAG schedules, creates task instances, and hands them to the executor.

3. **Executor & Workers**  
   Run the actual tasks, using local processes, Celery workers, Kubernetes pods, etc.

4. **Web Server**  
   A Flask-based UI for exploring DAGs, viewing logs, and triggering runs manually.

## Installation

### Prerequisites

- Python 3.8+ (check Airflow docs for supported versions).  
- A database for metadata (SQLite for local dev, Postgres/MySQL for production).  
- (Optional) Message broker like Redis/RabbitMQ if using the Celery executor.

### Basic installation (local/dev)

Airflow has strict version/constraints requirements; in production follow the official install docs. For a quick local setup:

```bash
pip install "apache-airflow[postgres,celery]==2.9.0" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.0/constraints-3.10.txt"
```

Then initialize the database and create a user (example for local testing, not run here):

```bash
airflow db init
airflow users create \
  --username admin --firstname Admin --lastname User \
  --role Admin --email admin@example.com
```

In [ ]:
# Quick install helper for notebooks (for experimentation only; prefer official docs)
# !pip install "apache-airflow==2.9.0" --constraint \
#   "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.0/constraints-3.10.txt"

## Basic Usage

### Quick start: a simple DAG with `PythonOperator`

Airflow loads DAGs from Python files in the **DAGs folder**. A minimal DAG:

- Runs once per day.  
- Has two tasks (`print_date` and `train_model`).  
- Shows basic dependency management (`print_date >> train_model`).

In [ ]:
# Minimal Airflow DAG example (not executed here)

from datetime import datetime

from airflow import DAG
from airflow.operators.python import PythonOperator


def print_date():
    """Example task that prints the current date."""
    from datetime import datetime
    print(f"Current date: {datetime.utcnow().isoformat()}Z")


def train_model():
    """Placeholder for a model training step."""
    # Your training logic here (call training script, kick off job, etc.)
    print("Training model...")


with DAG(
    dag_id="example_ml_pipeline",
    schedule_interval="0 2 * * *",  # every day at 02:00
    start_date=datetime(2023, 1, 1),
    catchup=False,
    default_args={"retries": 1},
    tags=["ml", "example"],
) as dag:

    t1 = PythonOperator(
        task_id="print_date",
        python_callable=print_date,
    )

    t2 = PythonOperator(
        task_id="train_model",
        python_callable=train_model,
    )

    t1 >> t2  # set task dependency

## Advanced Features

- **Sensors**: Wait for external conditions (file arrival in S3, partition in a table, etc.) before proceeding.  
- **Task groups** and **subDAG-like patterns**: Organize complex DAGs.  
- **Dynamic task mapping**: Create tasks at runtime based on a list of inputs (e.g., per dataset or per region).  
- **KubernetesExecutor & K8sPodOperator**: Run tasks in isolated Kubernetes pods.  
- **REST API & CLI**: Programmatically manage DAG runs and tasks.

These capabilities make Airflow suitable for complex, parameterized ML pipelines.

In [ ]:
# Sketch: dynamic task mapping (Airflow 2.x style, not executed here)

from airflow.decorators import dag, task
from datetime import datetime


@dag(schedule_interval=None, start_date=datetime(2023, 1, 1), catchup=False)
def dynamic_scoring():
    @task
    def score_dataset(dataset: str):
        print(f"Scoring dataset: {dataset}")
        # Call your scoring code here

    datasets = ["customers", "transactions", "events"]
    score_dataset.expand(dataset=datasets)


dynamic_scoring_dag = dynamic_scoring()

## Use Cases

- **ETL & data warehousing**: Orchestrate extract → transform → load pipelines into warehouses or data lakes.  
- **Feature engineering pipelines**: Build daily/weekly feature tables used by downstream ML models.  
- **Model training & retraining**: Trigger model training when new data arrives or on a schedule.  
- **Batch inference**: Run nightly scoring jobs generating predictions into tables or object storage.  
- **Orchestrating external systems**: Coordinate jobs across EMR, Databricks, Spark, Kubernetes, etc.

## Best Practices

1. **Keep DAG files lightweight**  
   - Import heavy libraries inside tasks, not at the module top level, to avoid slow DAG parsing.

2. **Separate orchestration from business logic**  
   - Put complex logic in **Python packages / modules** and call them from operators.

3. **Use clear naming & tagging**  
   - Name DAGs and tasks descriptively; use tags for ownership and domain.

4. **Set reasonable retries and SLAs**  
   - Avoid infinite retries; set SLAs for alerting when tasks run too long.

5. **Template connections & configuration**  
   - Use Airflow Connections, Variables, and environment-specific configs rather than hardcoding credentials or URLs.

## Common Pitfalls

1. **Putting heavy work in the scheduler**  
   - Symptom: Slow DAG parsing, scheduler lag.  
   - Fix: Move heavy imports and computations inside tasks.

2. **Overly large, monolithic DAGs**  
   - Symptom: Hard-to-maintain DAGs with many tasks and branches.  
   - Fix: Break into multiple DAGs, use clear boundaries and task groups.

3. **Ignoring time zones**  
   - Symptom: Confusing schedules and backfills across regions.  
   - Fix: Standardize on UTC and document expectations.

4. **Using Airflow for low-latency serving**  
   - Symptom: Trying to use Airflow as an online request/response system.  
   - Fix: Use Airflow for batch orchestration; use serving systems for online inference.

## Performance Optimization

- **Choose the right executor**:  
  - LocalExecutor for small setups, Celery/Kubernetes for scale.  
- **Tune parallelism and concurrency**:  
  - `parallelism`, `dag_concurrency`, and `max_active_runs_per_dag` in `airflow.cfg`.  
- **Use a robust metadata DB**:  
  - Postgres/MySQL with proper pooling and indexing.  
- **Offload heavy compute**:  
  - Use Spark, EMR, Databricks, or Kubernetes jobs for big computations, with Airflow orchestrating them.

In [ ]:
# Example airflow.cfg-style tuning (conceptual, not executable here)

airflow_cfg_snippet = """
[core]
parallelism = 32

[celery]
worker_concurrency = 16

[scheduler]
max_active_runs_per_dag = 4
"""

print(airflow_cfg_snippet)

## Production Deployment

- **Single-node / small teams**:  
  - Use LocalExecutor or CeleryExecutor with Docker Compose or a small VM.

- **Kubernetes**:  
  - Use the official Helm chart or platforms like **Astronomer**.  
  - Leverage KubernetesExecutor or K8sPodOperator for per-task isolation.

- **Managed offerings**:  
  - Cloud providers and vendors offer managed Airflow services; these can simplify upgrades and scaling.

Key concerns: backups of the metadata DB, secrets management, and rolling upgrades with minimal downtime.

## Monitoring and Observability

- **Airflow UI**:  
  - DAG graph, Gantt chart, task instance details, and logs per task.

- **Metrics**:  
  - Integrate with Prometheus/Grafana for scheduler/task metrics.  

- **Logging**:  
  - Ship logs to S3/GCS/CloudWatch/Stackdriver/Elastic for long-term analysis.  

- **Alerting**:  
  - Use email, Slack, PagerDuty, or other notifiers for task failures or SLA misses.

## Troubleshooting

- **DAG not showing up in UI**:  
  - Check for syntax errors in the DAG file; view scheduler logs; ensure file is in the DAGs folder.

- **Tasks stuck in `queued`**:  
  - Inspect executor/worker logs; verify workers are running and have capacity.

- **Frequent task failures**:  
  - Check logs for exceptions; adjust retries; verify external dependencies (DBs, cloud services).

- **Scheduler performance issues**:  
  - Profile DAG parsing time; simplify or split heavy DAGs; scale scheduler resources.

## Comparison with Alternatives

| Aspect | Airflow | Argo Workflows | Kubeflow Pipelines |
|--------|---------|----------------|---------------------|
| Platform | VM / Kubernetes | Kubernetes-native CRD | Kubernetes-native CRD |
| Workflows defined as | Python code | YAML (K8s manifests) | Python DSL / YAML |
| Best for | Batch ETL & ML pipelines | Container-native workflows | ML-specific pipelines |
| UI & ecosystem | Very mature | Mature | ML-focused |

Choose Airflow when you:

- Prefer **Python DAGs** and strong scheduling/backfill semantics.  
- Are orchestrating **many heterogeneous data/ML tasks**, not just containerized steps.  
- Want to leverage the large ecosystem of Airflow providers.

## Resources

- Official docs: https://airflow.apache.org/docs/  
- Concepts & tutorial: https://airflow.apache.org/docs/apache-airflow/stable/tutorial/index.html  
- Providers (integrations): https://airflow.apache.org/docs/apache-airflow-providers/index.html  
- GitHub repo: https://github.com/apache/airflow

Community:

- Slack: https://apache-airflow-slack.herokuapp.com/  
- Stack Overflow: https://stackoverflow.com/questions/tagged/airflow  
- Mailing lists & release notes: linked from the official site.